In [53]:
import numpy as np
import pandas as pd

df = pd.read_csv('/content/diabetes_dataset.csv')

In [54]:
df.isnull().sum()

,0
patient_id,0
age,0
gender,0
pregnancies,0
bmi,2250
skin_thickness_mm,0
sbp_mmhg,0
dbp_mmhg,0
glucose_mg_dl,0
fasting_glucose_mg_dl,0


In [55]:
df['bmi'] = df['bmi'].copy()


In [56]:
df['insulin_mu_l'] = df['insulin_mu_l'].copy()

In [57]:
df['bmi_ran'] = df['bmi']
s = df['bmi'].dropna().sample(df['bmi'].isnull().sum())
s.index = df[df['bmi'].isnull()].index
df.loc[df['bmi'].isnull(), 'bmi_ran'] = s

In [58]:
df['insulin_mu_l_ran'] = df['insulin_mu_l']
s = df['insulin_mu_l'].dropna().sample(df['insulin_mu_l'].isnull().sum())
s.index = df[df['insulin_mu_l'].isnull()].index
df.loc[df['insulin_mu_l'].isnull(), 'insulin_mu_l_ran'] = s

In [59]:
df.columns

Index(['patient_id', 'age', 'gender', 'pregnancies', 'bmi',
       'skin_thickness_mm', 'sbp_mmhg', 'dbp_mmhg', 'glucose_mg_dl',
       'fasting_glucose_mg_dl', 'ogtt_2hr_mg_dl', 'hba1c_pct', 'insulin_mu_l',
       'ldl_mg_dl', 'hdl_mg_dl', 'triglycerides_mg_dl', 'creatinine_mg_dl',
       'physical_activity', 'smoking_status', 'family_history_diabetes',
       'hypertension', 'diagnosis', 'diagnosis_code', 'bmi_ran',
       'insulin_mu_l_ran'],
      dtype='object')

In [60]:
df = df.drop(['patient_id', 'bmi', 'insulin_mu_l', 'diagnosis'], axis = 1)

In [61]:
import sklearn
from sklearn.model_selection import train_test_split

In [62]:
X = df.drop('diagnosis_code', axis=1)

y = df['diagnosis_code']

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [64]:
X_train_nums_cols = X_train.select_dtypes(exclude='object')

X_train_cats_cols = X_train.select_dtypes(include='object')

X_test_nums_cols = X_test.select_dtypes(exclude='object')

X_test_cats_cols = X_test.select_dtypes(include='object')

In [65]:
print("X_train_nums_cols :", X_train_nums_cols.shape)

print("X_train_cats_cols :", X_train_cats_cols.shape)

print("X_test_nums_cols :", X_test_nums_cols.shape)

print("X_test_cats_cols :", X_test_cats_cols.shape)

X_train_nums_cols : (12000, 17)
X_train_cats_cols : (12000, 3)
X_test_nums_cols : (3000, 17)
X_test_cats_cols : (3000, 3)


In [66]:
import numpy as np
from scipy import stats
from sklearn.preprocessing import PowerTransformer

# Extreme skewed numerical columns only
column = ['skin_thickness_mm', 'sbp_mmhg', 'dbp_mmhg','glucose_mg_dl',
          'fasting_glucose_mg_dl', 'ogtt_2hr_mg_dl', 'hba1c_pct','ldl_mg_dl',
          'hdl_mg_dl', 'triglycerides_mg_dl', 'creatinine_mg_dl', 'bmi_ran',
          'insulin_mu_l_ran']

# Yeo-Johnson Transformer
yj = PowerTransformer(method='yeo-johnson')

for i in column:

    # 1. Log Transformation
    X_train_nums_cols[i + '_log'] = np.log1p(X_train_nums_cols[i])

    # 2. Reciprocal Transformation
    X_train_nums_cols[i + '_reciprocal'] = 1 / (X_train_nums_cols[i] + 1)

    # 3. Box-Cox Transformation
    if (X_train_nums_cols[i] > 0).all():
        X_train_nums_cols[i + '_boxcox'], _ = stats.boxcox(X_train_nums_cols[i])
    else:
        shifted_data = X_train_nums_cols[i] + abs(X_train_nums_cols[i].min()) + 1
        X_train_nums_cols[i + '_boxcox'], _ = stats.boxcox(shifted_data)

    # 4. Yeo-Johnson Transformation
    X_train_nums_cols[i + '_yeojohnson'] = yj.fit_transform(X_train_nums_cols[[i]])

    # 5. Square Root Transformation
    X_train_nums_cols[i + '_sqrt'] = np.sqrt(X_train_nums_cols[i])

    # 6. Cube Transformation
    X_train_nums_cols[i + '_cube'] = np.power(X_train_nums_cols[i], 3)

print("Transformations Applied Successfully")

Transformations Applied Successfully


In [68]:
X_train_nums_cols.columns

Index(['age', 'pregnancies', 'skin_thickness_mm', 'sbp_mmhg', 'dbp_mmhg',
       'glucose_mg_dl', 'fasting_glucose_mg_dl', 'ogtt_2hr_mg_dl', 'hba1c_pct',
       'ldl_mg_dl', 'hdl_mg_dl', 'triglycerides_mg_dl', 'creatinine_mg_dl',
       'family_history_diabetes', 'hypertension', 'bmi_ran',
       'insulin_mu_l_ran', 'skin_thickness_mm_log',
       'skin_thickness_mm_reciprocal', 'skin_thickness_mm_boxcox',
       'skin_thickness_mm_yeojohnson', 'skin_thickness_mm_sqrt',
       'skin_thickness_mm_cube', 'sbp_mmhg_log', 'sbp_mmhg_reciprocal',
       'sbp_mmhg_boxcox', 'sbp_mmhg_yeojohnson', 'sbp_mmhg_sqrt',
       'sbp_mmhg_cube', 'dbp_mmhg_log', 'dbp_mmhg_reciprocal',
       'dbp_mmhg_boxcox', 'dbp_mmhg_yeojohnson', 'dbp_mmhg_sqrt',
       'dbp_mmhg_cube', 'glucose_mg_dl_log', 'glucose_mg_dl_reciprocal',
       'glucose_mg_dl_boxcox', 'glucose_mg_dl_yeojohnson',
       'glucose_mg_dl_sqrt', 'glucose_mg_dl_cube', 'fasting_glucose_mg_dl_log',
       'fasting_glucose_mg_dl_reciprocal', 'fa

In [69]:
# Comparing Original and Transformed Column Skewness

for i in column:

    print(f"\n================ {i} ================\n")

    # Original Column
    print("Original :", X_train_nums_cols[i].skew())

    # Log
    print("Log :", X_train_nums_cols[i + '_log'].skew())

    # Reciprocal
    print("Reciprocal :", X_train_nums_cols[i + '_reciprocal'].skew())

    # Box-Cox
    print("BoxCox :", X_train_nums_cols[i + '_boxcox'].skew())

    # Yeo-Johnson
    print("YeoJohnson :", X_train_nums_cols[i + '_yeojohnson'].skew())

    # Square Root
    print("Sqrt :", X_train_nums_cols[i + '_sqrt'].skew())

    # Cube
    print("Cube :", X_train_nums_cols[i + '_cube'].skew())


================ skin_thickness_mm ================

Original : 0.057176127226884574
Log : -2.431059692331758
Reciprocal : 7.82794425340209
BoxCox : -0.07230422107135608
YeoJohnson : -0.07230418404028663
Sqrt : -1.0000463838974814
Cube : 2.1536600953820395

================ sbp_mmhg ================

Original : 0.0044301545316126996
Log : -0.4361358375151363
Reciprocal : 0.9349065004665236
BoxCox : -0.00240221918117387
YeoJohnson : -0.0023755781959302388
Sqrt : -0.21271534202353987
Cube : 0.842679584913894

================ dbp_mmhg ================

Original : -0.010656028011425562
Log : -0.512668758774643
Reciprocal : 1.1122148291028564
BoxCox : 0.0031864938697280932
YeoJohnson : 0.0031574586812120557
Sqrt : -0.25667597835440986
Cube : 0.9094636487744165

================ glucose_mg_dl ================

Original : 0.22928636310931763
Log : -0.4705560222300154
Reciprocal : 1.1331500339816383
BoxCox : -0.040290841789285804
YeoJohnson : -0.03960827139389914
Sqrt : -0.1277861692950164
C